# 4. Deriving Scale Factors
Tutorial on deriving b tagging scale factors (SF) to correct the discriminator shape in Monte Carlo simulation.

The scale factors to correct b jets are derived in a $t\bar{t}$ phase space, they are derived differential in the jet's transverse momentum to avoid the sculpting of this variable.
Due to the short time available in this tutorial and to avoid waiting for processing, the distributions to measure the SFs are provided in form of histograms of the b tag score for multiple bins of the transverse momentum.



First import the packeges we will need:
- uproot to read in the histograms
- numpy for calculations
- matplotlib for plotting
- scipy for the interpolation
- correctionlib for saving the scale factors in a common format used in CMS

In [ ]:
import uproot

In [ ]:
import numpy as np

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
from scipy.interpolate import PchipInterpolator, PPoly

In [ ]:
import correctionlib.schemav2 as cs

Read in the histograms for the differnet $p_T$ bins, for data and MC (splitted into light and heavy falvour).

In the name of the individual histograms you can also find information about $|\eta|$ and $\Delta R$, variables the in which the SFs can be differential, but these are not used in this tutorial. The $p_T$ edges used for this tuturial are defined below.

In [ ]:
hists = uproot.open("/work/projects/hats2024/tagging/hists.root")

# On lxplus, use path /eos/cms/store/group/phys_btag/PODAS/Exercise3/

In [ ]:
hists.keys()

In [ ]:
pt_edges = [(20,30), (30,50), (50,70), (70,100), (100,140), (140, np.inf)]
pt_bins = [f"{lower}to{upper}" for lower, upper in pt_edges]
flavors = ["heavy", "light", "data"]
def get_data(pt_bin: str, flavor: str):
    return hists.get(f"HF_deepjet_pt{pt_bin}_eta0to2.5_dr0toinf_{flavor};1")

After reading in the historagams it's your turn to start with the SF derivation. Your first tasks will be:

1. to normalise the MC to the data yield in each $p_T$ bin (we only want to correct the MC shape not the MC yield)
2. to plot the data to MC agreement for each $p_T$ bin:
    - you can use bar(x, y, width=...) for the MC to produce a plot similar to a histogram
    - calculate the ratio between data and MC and plot it below the histogram

In [ ]:
for pt_bin in pt_bins:
    data_values, data_edges = get_data(pt_bin, "data").to_numpy()
    heavy_values = get_data(pt_bin, "heavy").values().copy()
    light_values = get_data(pt_bin, "light").values().copy()    ## add your code here 

Now we will start to derive the SFs for each $p_T$ bin individually. As a first step you can calculate the SFs from the histogram counts provided in the following way:

1. First we have to normalise again to only account for shape changing effects in our SFs.
2. As we want to derive SFs for heavy (b) flavoured jets, we need to deal with the light (light + c) flavour contamination. To do so we will subtract the MC prediction for the contamination from data, to get an estimate of the heavy flavour contribution in data.$^*$
3. Now we can calculate the factor by which we have to scale the heavy MC to match the contamination free data. This is done for each bin in the b tag score and each $p_T$ bin.
4. To create a smooth SF we will now interpolate the SFs between the b tag score bins. You can use the [<tt>PchipInterpolator</tt>](https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.PchipInterpolator.html) for a piecewise polynomial interpolation to cubic order. You can get the coefficients and breakpoints in the power basis with [<tt>PPoly</tt>](https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.PPoly.html#scipy.interpolate.PPoly). 

    **Hint**: You can use: <tt>interpolator = PchipInterpolator()</tt> and <tt>PPoly(interpolator.c, interpolator.x)</tt>
    
    **Question**: How do we describe the very first and very last bin?
    
5. Write the formula to calculate the SFs in term of coefficients and breakpoints. Use the notation: $(x >= x_{min})*(x < x_{max})*[c_0(x - x_{BP})*(x - x_{BP})*(x - x_{BP}) + c_1(x - x_{BP})*(x - x_{BP})\,+\, ...]$ for the bin centers $x_{min}$ and the bin center of the following bin $x_{max}$ and the breakpoints of the interpolater $x_{BP}$. The first term makes sure you are only apply the piece wise interpolation to the correct bin by a heavyside function, the second form represents the interpolation. Make sure to get your parentheses right :)
6. Now you can plot the SFs together with the interpolation.

$*$ Of course we can expect the light flavour contribution to also have some shape differences between data and MC, such that the contamination would also be affected by some light flavour SFs. This is why these SFs are usually derived in an iterative procedure, in which first heavy and light SFs are derived separately and in the second iteration the contamination will be corrected by the SFs from the first iteration.

In [ ]:
functions = []
for pt_bin in pt_bins:
    data_values, data_edges = get_data(pt_bin, "data").to_numpy()
    heavy_values = get_data(pt_bin, "heavy").values().copy()
    light_values = get_data(pt_bin, "light").values().copy()
    
    
    
    # writing the piecewise function:
    # piecewise function
    function_pieces = []
    # left side (Heavyside function for first bin)
    
    # intermediate (degree 3 polynomial, power basis)
    # some code to help write the function
    # for interpolator_idx, (x_min, x_max) in enumerate(zip(centers, centers[1:])):
    #     interpolator_coefficients = ppoly_interpolator.c[:, interpolator_idx]
    #     if any(np.isnan(interpolator_coefficients)):
    #         raise ValueError("One or more coefficient(s) is NaN!")
    #     interpolator_x = ppoly_interpolator.x[interpolator_idx] # breakpoints
    #     add the function here:
    #
    #     multiply by heavyside function
    #     function_pieces.append(f"(x >= {x_min}) * (x < {x_max}) * ({function_piece})")
    # right side (Heavyside function for last bin)
    
    # join pieces
    functions.append(" + ".join(function_pieces))

The last step is to save the derived correction in the correctionlib json format. For this you can follow the [Correctionlib tutorial](https://cms-nanoaod.github.io/correctionlib/correctionlib_tutorial.html#Correctionlib-tutorial) and combine different examples for writing binned and formula based correction.

After defining your correction you can print and test it with <tt>corr.to_evaluator()</tt>. If your tests work out fine, you can save your correction as described in the tutorial.   

In [ ]:
# export to correctionlib


In [ ]:
# print correction

In [ ]:
# check if your correction is working


In [ ]:
# writing it all out (save you correction)
